# Steering SDM — endpoint WR KR260 (fw v6)

Notebook per il **firmware endpoint WR `kr260_wr_sdm v6`** (md5 `dd0b03d0`) — da
usare in lab per verificare lo steering e monitorare il lock del softpll.
Differenze rispetto a `notebooks/freq_meter.ipynb` (esercizio standalone):

| | esercizio standalone | endpoint WR v6 (questo) |
|---|---|---|
| steering | scrittura DRP `QPLL0_SDM_CFG*` | **base via EMIO** (porte `SDM0DATA/TOGGLE` del GT) |
| clock contato dal fmeter | TXPRGDIVCLK (~161 MHz) | **TXUSRCLK2 = 62.5 MHz** |
| clock del gate | pl_clk0 = 100 MHz | **pl_clk0 = 124.99875 MHz** (gate 100e6 = 0.800 s) |
| linea | 10.3125 Gbps, FBDIV 66 | 1.25 Gbps 8b10b HW, QPLL0 fracN FBDIV 64 |

## Mappa AXI
| Base | Periferica |
|------|------------|
| 0xA0020000 | DRP GTHE4_COMMON (QPLL0) |
| 0xA0030000 | Frequenzimetro (TXUSRCLK2) |
| 0xA0040000 | debug_bridge (XVC) |

## EMIO (gpiochip zynqmp base 334; EMIO n = gpio 412+n)
| gpio | Segnale |
|------|---------|
| 466..490 | `sdm_base_i[24:0]` (out) |
| 426 | `phy_rdy` (in) |
| 428..434 | `sdm_word[24:18]` mirror (in) |

⚠️ **Trappole**: mai toccare 0xA00xxxxx senza l'app caricata (hang AXI);
da CLI usare `busybox devmem ADDR 32` (devmem2 fa accessi a 64 bit → SIGBUS).
Gli accessi mmap di questo notebook sono a 32 bit e sono sicuri.
I gpio sysfs richiedono root: lanciare Jupyter con sudo, oppure eseguire
prima `sudo bash sdm_steer_test.sh` che esporta i gpio (poi `chown`/`chmod`).

In [ ]:
import mmap, struct, time

# Indirizzi fisici AXI (fw kr260_wr_sdm v6)
FC_BASE   = 0xA003_0000   # frequenzimetro
COMM_BASE = 0xA002_0000   # DRP GTHE4_COMMON (QPLL0)
PAGE_SIZE = 0x1000

# Frequenzimetro: +0 GATE, +4 FREQ_CNT, +8 STATUS
GATE_OFF, CNT_OFF, STAT_OFF = 0x00, 0x04, 0x08
# DRP bridge: +0 ADDR, +4 DI, +8 DO, +C EN, +10 WE, +14 RDY
DRP_ADDR_OFF, DRP_DI_OFF, DRP_DO_OFF = 0x00, 0x04, 0x08
DRP_EN_OFF, DRP_WE_OFF, DRP_RDY_OFF  = 0x0C, 0x10, 0x14

def open_axi(base, size=PAGE_SIZE):
    f = open('/dev/mem', 'r+b')
    mm = mmap.mmap(f.fileno(), size, offset=base)
    return f, mm

def rd32(mm, off):
    return struct.unpack_from('<I', mm, off)[0]

def wr32(mm, off, val):
    struct.pack_into('<I', mm, off, val & 0xFFFFFFFF)

f_fc, mm_fc = open_axi(FC_BASE)
f_dc, mm_dc = open_axi(COMM_BASE)
print('mmap ok: fmeter @%08X, drp common @%08X' % (FC_BASE, COMM_BASE))

In [ ]:
# --- Frequenzimetro: gate in cicli di pl_clk0 = 124.99875 MHz ---
PL_CLK0_HZ = 124_998_750          # misurato (NON 100 MHz!)
F_NOM_HZ   = 62_500_000           # TXUSRCLK2 nominale

def fmeter(gate_seconds=0.8):
    gate_cycles = int(round(gate_seconds * PL_CLK0_HZ))
    wr32(mm_fc, GATE_OFF, gate_cycles)
    deadline = time.time() + gate_seconds + 5.0
    time.sleep(gate_seconds + 0.2)
    while time.time() < deadline:
        if rd32(mm_fc, STAT_OFF) & 1:
            cnt = rd32(mm_fc, CNT_OFF)
            return cnt, cnt / gate_seconds
        time.sleep(0.05)
    raise TimeoutError('fmeter: nessun risultato')

cnt, f = fmeter()
print(f'FREQ_CNT={cnt}  ->  TXUSRCLK2 = {f/1e6:.6f} MHz  (Δ = {(f-F_NOM_HZ):+.1f} Hz, {(f/F_NOM_HZ-1)*1e6:+.3f} ppm)')

In [ ]:
# --- Base SDM via EMIO (richiede root sui gpio sysfs) ---
GPIO_BASE_OUT = 466   # sdm_base bit0
GPIO_PHY_RDY  = 426
GPIO_WORD_HI  = 428   # sdm_word[18]; ..434 = [24]

def _gpio(path):  return '/sys/class/gpio/' + path
def _export(g):
    import os
    if not os.path.isdir(_gpio(f'gpio{g}')):
        open(_gpio('export'), 'w').write(str(g))

def set_base(v):
    """Imposta sdm_base_i[24:0]. Zona lineare verificata: 0 .. ~2^22 (+3660 ppm).
    Saturazione VCO a ~+6000 ppm (out_sdm12)."""
    assert 0 <= v < 2**25
    for i in range(25):
        g = GPIO_BASE_OUT + i
        _export(g)
        open(_gpio(f'gpio{g}/direction'), 'w').write('out')
        open(_gpio(f'gpio{g}/value'), 'w').write(str((v >> i) & 1))

def read_status():
    _export(GPIO_PHY_RDY)
    open(_gpio(f'gpio{GPIO_PHY_RDY}/direction'), 'w').write('in')
    phy = int(open(_gpio(f'gpio{GPIO_PHY_RDY}/value')).read())
    w = 0
    for i in range(7):
        g = GPIO_WORD_HI + i
        _export(g)
        open(_gpio(f'gpio{g}/direction'), 'w').write('in')
        w |= int(open(_gpio(f'gpio{g}/value')).read()) << i
    return {'phy_rdy': phy, 'word_24_18': w}

print(read_status())

In [ ]:
# --- Lettura DRP QPLL0 (sanity: FBDIV @0x14 -> 0x3E = N-2 -> N=64) ---
def drp_read(addr):
    wr32(mm_dc, DRP_ADDR_OFF, addr)
    wr32(mm_dc, DRP_WE_OFF, 0)
    wr32(mm_dc, DRP_EN_OFF, 1)
    for _ in range(1000):
        if rd32(mm_dc, DRP_RDY_OFF) & 1:
            return rd32(mm_dc, DRP_DO_OFF)
    raise TimeoutError('DRP rdy timeout')

fbdiv = drp_read(0x14)
print(f'QPLL0 FBDIV raw = 0x{fbdiv:04X}  (atteso 0x003E -> N = {fbdiv + 2})')

In [ ]:
# --- Sweep steering: base -> frequenza (zona lineare) + grafico ---
import matplotlib.pyplot as plt

bases = [0, 2**19, 2**20, 2**21, 2**21 + 2**20, 2**22, 0]
rows = []
for b in bases:
    set_base(b); time.sleep(0.5)
    st = read_status()
    cnt, f = fmeter()
    ppm = (f / F_NOM_HZ - 1) * 1e6
    rows.append((b, st['word_24_18'], st['phy_rdy'], cnt, f, ppm))
    print(f"base={b:>9}  word[24:18]={st['word_24_18']:>3}  phy_rdy={st['phy_rdy']}  "
          f"f={f/1e6:.6f} MHz  ({ppm:+9.3f} ppm)")

xs  = [r[0] for r in rows[:-1]]
ys  = [r[5] for r in rows[:-1]]
exp = [b / 2**24 / 64 * 1e6 for b in xs]   # atteso (DAC al centro)
plt.figure(figsize=(7,4))
plt.plot(xs, ys, 'o-', label='misurato')
plt.plot(xs, exp, 's--', label='teorico base/2^24/64')
plt.xlabel('sdm_base'); plt.ylabel('Δf [ppm]'); plt.grid(True); plt.legend()
plt.title('Steering QPLL0 SDM — KR260 fw v6')
plt.tight_layout(); plt.savefig('sdm_sweep.png', dpi=120); plt.show()
print('NB: offset costante ~ -(2^18)/2^24/64 * 1e6 = -244 ppm/16 se il softpll ha scritto DAC=0')

In [ ]:
# --- LAB: monitor del lock (softpll che sterza contro il partner WR) ---
# Da usare con base=0 e sessione WR attiva: il DAC del softpll muove sdm_word.
# word[24:18] != 0 e variabile nel tempo = il softpll sta sterzando.
import IPython.display

set_base(0)
try:
    while True:
        st = read_status()
        cnt, f = fmeter(0.4)
        ppm = (f / F_NOM_HZ - 1) * 1e6
        IPython.display.clear_output(wait=True)
        print(time.strftime('%H:%M:%S'),
              f"phy_rdy={st['phy_rdy']}  word[24:18]={st['word_24_18']:>3}  "
              f"f={f/1e6:.6f} MHz  ({ppm:+8.3f} ppm)")
except KeyboardInterrupt:
    print('monitor fermato')

In [ ]:
# --- Cleanup ---
mm_fc.close(); f_fc.close()
mm_dc.close(); f_dc.close()
print('mmap chiuse')